# Map boost dose check (preregistered: `docs/preregistration_map_boost_dose.md`)

Was the map boost too small? Same Gemma 3 12B (8-bit, as in the boost run), same map directions and random controls
(reused from the boost run's files on Drive). Adds (1) a positive control: a push of the boost's own size along the
direction that favours the right name, and (2) bigger boosts (×16, ×32, ×64).

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. Runtime → Change runtime type → **L4** or **A100** (either works; the model is always loaded in 8-bit, like the
   boost run, so its saved files can be reused). About 1–1.5 hours. Then **Run all**.

Results are saved to Drive as they are produced; after a disconnect, just press Run all again (finished conditions are
skipped). The last cell downloads `map_boost_dose_results.zip`; send that file back.


In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> L4 / A100"
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert GB > 20, f"Only {GB:.0f} GB of GPU memory: pick an L4 or A100"
GEMMA, BS = "gemma3-12b-8bit", 16      # the boost run's weights and batch size, on any GPU (its files are reused)
print(f"{GB:.0f} GB: {GEMMA}, batch size {BS}")


In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/map_boost_dose.py"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
os.makedirs(f"{RES}/map_boost_dose", exist_ok=True)
for f in ["gemma3-12b-8bit_subspaces.pt", "gemma3-12b-8bit_items.csv"]:
    print("boost run file", f, "found" if os.path.exists(f"{RES}/map_boost/{f}") else "MISSING (will be recomputed: slower)")
print("already saved:", sorted(os.listdir(f"{RES}/map_boost_dose")))

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' imports); bitsandbytes for 8-bit Gemma
!pip -q install "transformers==4.57.3" nnsight accelerate bitsandbytes
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Map boost dose] Colab run started (or resumed)")
# Which gated models can this token open? (the base models need their own licence acceptance)
from huggingface_hub import hf_hub_download
for repo in ["google/gemma-3-12b-it"]:
    try:
        hf_hub_download(repo, "config.json", token=os.environ["HF_TOKEN"])
        print("ok     ", repo)
    except Exception as e:
        print("NO ACCESS", repo, "-> accept the licence on its Hugging Face page, then rerun this cell", type(e).__name__)


In [ ]:
# 5. The run: reuse the boost run's directions and answers -> positive control + larger boosts -> report
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/map_boost_dose.py --model {GEMMA} --batch-size {BS} 2>&1 | grep -v -i warning


In [ ]:
# 6. One zip of the results + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/map_boost_dose_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/map_boost_dose/*"):
        if not f.endswith(".pt"):
            z.write(f, "results/map_boost_dose/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
    print("\n".join(z.namelist()))
shutil.copy("/content/map_boost_dose_results.zip", f"{D}/map_boost_dose_results.zip")
notify("[Map boost dose] zip saved: MyDrive/geomap/map_boost_dose_results.zip")
from google.colab import files
files.download("/content/map_boost_dose_results.zip")
